In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :reciprocal

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 2

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [reciprocal_model] Fitting chain 2 (tau=34)
[ Info: [reciprocal] iter 1000/1000000 elapsed=3.9s, rate=0.125, mean=[1.721, 0.00087, 1.044, 0.290], std=[0.2904, 0.000358, 0.0612, 0.0980] [ADAPT]
[ Info: [reciprocal] iter 2000/1000000 elapsed=6.9s, rate=0.095, mean=[1.934, 0.00081, 1.053, 0.225], std=[0.2819, 0.000267, 0.0475, 0.0905] [ADAPT]
[ Info: [reciprocal] iter 3000/1000000 elapsed=9.1s, rate=0.087, mean=[2.001, 0.00078, 1.077, 0.213], std=[0.2452, 0.000227, 0.0492, 0.0756] [ADAPT]
[ Info: [reciprocal] iter 4000/1000000 elapsed=11.3s, rate=0.082, mean=[2.045, 0.00078, 1.111, 0.204], std=[0.2239, 0.000202, 0.0693, 0.0672] [ADAPT]
[ Info: [reciprocal] iter 5000/1000000 elapsed=13.5s, rate=0.084, mean=[2.061, 0.00078, 1.145, 0.200], std=[0.2045, 0.000186, 0.0894, 0.0609] [ADAPT]
[ Info: [reciprocal] iter 6000/1000000 elapsed=15.7s, rate=0.080, mean=[2.066, 0.00078, 1.175, 0.195], std=[0.1873, 0.000175, 0.1014, 0.0563] [ADAPT]
[ Info: [reciprocal] iter 7000/1000000 elapsed=17.9